# Delta table lokálisan, Databricks és Synapse nélkül

Ez a notebook azt mutatja meg, hogy egy **Delta table** nem feltétlenül Databricksben vagy Synapse-ban jön létre. A Delta Lake egy nyílt tárolási formátum: ha van hozzá kompatibilis writer, akkor lokális fájlrendszerre is tudsz Delta táblát írni.

A notebook fő gondolata:

```text
Delta table = Parquet adatfájlok + _delta_log tranzakciós napló
Metastore/Catalog = névvel ellátott bejegyzés, ami rámutat a Delta table locationre
```

Itt nem használunk Databrickset, Synapse-t vagy Azure-t. Minden lokálisan történik.

A notebook későbbi részei azt is megmutatják, hogyan kapcsolódik ehhez az ETL/ELT, a bronze-silver-gold rétegezés, valamint a fact/dimension modellezés.

## 0. Fogalmi térkép: Data Lake, Data Warehouse, Lakehouse, Metastore

Mielőtt létrehozunk egy lokális Delta table-t, érdemes elhelyezni a példát a nagyobb data engineering térképen.

| Fogalom | Röviden | Adat | Erősség | Tipikus használat |
|---|---|---|---|---|
| **Data Warehouse** | Üzleti riporting és konzisztens SQL elemzés | Tisztított, modellezett, strukturált | Gyors BI, erős SQL, governance | Pénzügyi riport, vezetői dashboard, data mart |
| **Data Lake** | Nyers és sokféle adat olcsó, skálázható tárolása | Fájlok: JSON, CSV, Parquet, képek, logok | Rugalmasság, nagy skála, alacsony storage költség | Raw landing zone, archívum, data science előkészítés |
| **Lakehouse** | Lake rugalmasság warehouse-szerű táblakezeléssel | Nyílt table formatok objektumtáron | ACID, time travel, batch + streaming, ML + BI | Bronze-silver-gold platform, közös analitikai adatbázis |
| **Metastore** | Táblák, sémák és lokációk nyilvántartása | Metaadat, nem maga az üzleti adat | Név, schema, permission, discovery | Spark SQL táblák, catalog, jogosultságkezelés |

Ebben a notebookban a **lakehouse** gondolkodást próbáljuk ki kicsiben:

```text
Data Lake rész: lokális fájlok és mappák
Lakehouse rész: Delta table = Parquet + _delta_log
Metastore rész: most nincs tényleges metastore, csak bemutatjuk, mit adna hozzá
Data Warehouse rész: fact/dimension és gold réteg, vagyis riportbarát modell
```

A lényeg: a data lake főleg tárolási minta, a warehouse főleg riportolási és modellezési minta, a lakehouse a kettő közötti híd, a metastore pedig a név- és metadata-réteg.

## 1. Architektúra-térkép

A architektúra-térkép azt mutatja meg, hogyan áll össze egy tipikus data engineering platform a forrásrendszerektől a fogyasztásig. A notebookban ennek egy kicsinyített, lokális változatát építjük fel.

```text
Forrás -> Beérkeztetés -> Tárolás -> Feldolgozás -> Metaadat / Bizalom -> Fogyasztás
```

| Sáv | Tipikus komponensek | Mit jelent ebben a notebookban? |
|---|---|---|
| **Forrás** | OLTP adatbázis, API, fájl export, event stream, IoT | Kézzel létrehozott pandas DataFrame-ek és CSV fájlok szimulálják a forrásadatot |
| **Beérkeztetés** | Batch ingest, CDC, Kafka, Event Hub, landing zone | CSV-k beolvasása batch módban, plusz micro-batch streaming szimuláció |
| **Tárolás** | Object storage, data lake, Delta table, Iceberg table, warehouse | Lokális mappák, Parquet fájlok és Delta táblákat tartó könyvtárak |
| **Feldolgozás** | Spark, SQL engine, dbt, stream processing, materialized view | pandas + deltalake transzformációk: bronze -> silver -> gold |
| **Metaadat** | Metastore, data catalog, schema registry, lineage, data contract | A notebook elmagyarázza a metastore szerepet; tényleges catalogot most nem indítunk |
| **Bizalom** | Data quality, RBAC, masking, encryption, observability | A demóban csak alapszintű ellenőrzések vannak; production rendszerben ez külön réteg |
| **Fogyasztás** | BI dashboard, data mart, feature store, vector database, reverse ETL | Gold KPI tábla, fact/dimension modell és riportbarát aggregátumok |

Ugyanez a kis példára vetítve:

```text
Fájl export / forrásadat
  -> landing/orders/*.csv
  -> bronze Delta table
  -> silver Delta table
  -> gold KPI és fact/dimension Delta táblák
  -> riport / BI / SQL elemzésre kész adat
```

A HTML térképen szereplő komponensek közül ebben a notebookban nem mindent futtatunk ténylegesen. Például nincs Kafka, Spark cluster, RBAC vagy Data Catalog szerver. A lényeg az, hogy a **szerepeket** lásd: mi érkezik be, hol tároljuk, hol tisztítjuk, hol katalogizáljuk, és honnan fogyasztja az üzlet vagy az ML.


## 2. Notebook bootstrap: virtuális környezet és csomagok

Ez a notebook teljesen bootstrapelt: a projekt gyökerében lévő `bootstrap.ps1` vagy `bootstrap.sh` létrehozza a `.venv` virtuális környezetet, telepíti a `requirements.txt` csomagjait, és regisztrálja a Jupyter kernelt.

A cella notebookból is lefuttatja ugyanezt. Ha a notebook még nem a `.venv` kernelből fut, a csomagokat a jelenlegi kernelbe is telepíti, hogy kézi előkészítés nélkül végig lehessen futtatni.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REQUIRED_IMPORTS = ["deltalake", "pandas", "pyarrow"]
KERNEL_NAME = "de-delta-demo"
KERNEL_DISPLAY_NAME = "Python (.venv - Delta demo)"

project_dir = Path.cwd()
venv_dir = Path(".venv")

if os.name == "nt":
    venv_python = venv_dir / "Scripts" / "python.exe"
    bootstrap_cmd = [
        "powershell",
        "-NoProfile",
        "-ExecutionPolicy",
        "Bypass",
        "-File",
        str(project_dir / "bootstrap.ps1"),
    ]
else:
    venv_python = venv_dir / "bin" / "python"
    bootstrap_cmd = ["bash", str(project_dir / "bootstrap.sh")]

subprocess.check_call(bootstrap_cmd, cwd=project_dir)

running_python = Path(sys.executable).resolve()
target_python = venv_python.resolve()

if running_python != target_python:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-r", str(project_dir / "requirements.txt")])
    print("A notebook most nem a projekt .venv kernelből fut, ezért a csomagok a jelenlegi kernelbe is települtek.")

for module_name in REQUIRED_IMPORTS:
    __import__(module_name)

print(f"Virtuális környezet kész: {target_python}")
print(f"Kernel regisztrálva: {KERNEL_DISPLAY_NAME}")
print(f"Aktuális futó Python: {running_python}")

## 3. Környezet ellenőrzése

A példához a `deltalake` Python csomagot használjuk. Ez a delta-rs projekt Python bindingja, és Spark nélkül is tud Delta táblát írni/olvasni.

A szükséges csomagokat az előző bootstrap cella már telepítette; ez a cella csak ellenőrzi, hogy minden importálható-e.

In [ ]:
import sys

import deltalake
import pandas as pd
import pyarrow as pa

print(f"Python: {sys.executable}")
print(f"deltalake: {deltalake.__version__}")
print(f"pandas: {pd.__version__}")
print(f"pyarrow: {pa.__version__}")

## 4. Importok és lokális célmappa

A Delta table fizikailag egy mappa lesz a gépeden. Ebben lesznek a Parquet adatfájlok és a `_delta_log` könyvtár.

In [ ]:
from pathlib import Path
import json
import shutil

import pandas as pd
from deltalake import DeltaTable
from deltalake.writer import write_deltalake

base_dir = Path("delta_demo_output")
table_path = base_dir / "sales_delta"

if base_dir.exists():
    shutil.rmtree(base_dir)

base_dir.mkdir(parents=True, exist_ok=True)

table_path

## 5. Forrásadat: egy sima Pandas DataFrame

Ez lehetne CSV-ből, API-ból vagy adatbázisból jövő adat is. Most kézzel készítünk egy kis táblát.

In [ ]:
sales_df = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "customer_id": ["C-001", "C-002", "C-001", "C-003"],
    "amount": [12900, 4790, 8350, 21990],
    "order_date": pd.to_datetime(["2026-05-01", "2026-05-01", "2026-05-02", "2026-05-03"]),
    "source_system": ["webshop", "webshop", "pos", "webshop"]
})

sales_df

## 6. Delta table létrehozása lokálisan

Itt történik a lényeg. A `write_deltalake(...)` létrehozza a Delta table-t a megadott mappában.

Nem kell hozzá:

- Databricks
- Synapse
- Azure
- metastore
- cluster

Kell hozzá egy Delta-kompatibilis writer, jelen esetben a `deltalake` csomag.

In [ ]:
write_deltalake(str(table_path), sales_df)

print(f"Delta table létrehozva itt: {table_path.resolve()}")

## 7. Nézzük meg, mi jött létre a fájlrendszerben

A Delta table mappájában Parquet fájlokat és egy `_delta_log` mappát kell látnod.

In [ ]:
for path in sorted(table_path.rglob("*")):
    relative = path.relative_to(table_path)
    marker = "[DIR] " if path.is_dir() else "[FILE]"
    print(marker, relative)

Tipikusan ilyesmit fogsz látni:

```text
sales_delta/
  part-00000-....snappy.parquet
  _delta_log/
    00000000000000000000.json
```

Ez már valódi Delta table. Nem azért, mert van neve egy catalogban, hanem mert a mappa Delta formátumú: adatfájlok + tranzakciós napló.

## 8. A `_delta_log` tartalma

A `_delta_log` JSON fájlok írják le a tábla tranzakcióit. Itt van például a schema és az, hogy mely Parquet fájlok tartoznak a táblához.

In [ ]:
log_files = sorted((table_path / "_delta_log").glob("*.json"))
log_files

In [ ]:
first_log = log_files[0]

with first_log.open("r", encoding="utf-8") as f:
    log_lines = [json.loads(line) for line in f]

for entry in log_lines:
    print(json.dumps(entry, indent=2)[:1200])
    print("-" * 80)

A logban többféle bejegyzést láthatsz, például:

- `protocol`: milyen Delta protokollverzió kell az olvasáshoz/íráshoz
- `metaData`: tábla metaadatai, schema, formátum
- `add`: új Parquet fájl hozzáadása a táblaverzióhoz
- `commitInfo`: ki/milyen művelettel hozta létre a verziót

Ezért mondjuk, hogy a Delta table nem csak adatfájlok halmaza, hanem tranzakciós táblaréteg.

## 9. Delta table olvasása path alapján

Most nincs metastore. Nem tudjuk azt mondani, hogy `SELECT * FROM silver.sales`, mert nincs ilyen regisztrált táblanév.

De path alapján tökéletesen olvasható a Delta table.

In [ ]:
dt = DeltaTable(str(table_path))

dt.to_pandas()

## 10. Schema lekérdezése

A schema nem a storage accountból jön. A schema a Delta metadata része lett, amikor megírtuk a táblát.

In [ ]:
print(dt.schema().json())

Mentális modell:

```text
Lokális mappa vagy ADLS path
  ↓
Parquet adatfájlok
  +
_delta_log/metaData
  ↓
Delta table schema
```

## 11. Új adatok hozzáírása

Most appendelünk két új sort. Ettől új Parquet fájl és új Delta log verzió jön létre.

In [ ]:
new_sales_df = pd.DataFrame({
    "order_id": [1005, 1006],
    "customer_id": ["C-004", "C-002"],
    "amount": [15900, 6990],
    "order_date": pd.to_datetime(["2026-05-04", "2026-05-04"]),
    "source_system": ["pos", "webshop"]
})

write_deltalake(str(table_path), new_sales_df, mode="append")

dt = DeltaTable(str(table_path))
dt.to_pandas().sort_values("order_id")

## 12. Verziók és time travel

A Delta log miatt a tábla verziózott. Az első írás volt a 0. verzió, az append után létrejött az 1. verzió.

In [ ]:
for log_file in sorted((table_path / "_delta_log").glob("*.json")):
    print(log_file.name)

In [ ]:
version_0 = DeltaTable(str(table_path), version=0)
version_1 = DeltaTable(str(table_path), version=1)

print("0. verzió sorok száma:", len(version_0.to_pandas()))
print("1. verzió sorok száma:", len(version_1.to_pandas()))

Ez a **time travel** alapja: nem csak a legfrissebb állapotot tudod olvasni, hanem korábbi verziókat is, amíg a régi fájlokat nem törölted retention/vacuum miatt.

## 13. Hol van itt a metastore?

Ebben a notebookban **nincs metastore**. Csak egy Delta table van egy lokális path-on.

A metastore akkor jön képbe, amikor ehhez a path-hoz táblanevet akarsz rendelni.

Például Databricksben vagy Spark SQL-ben valami ilyesmit csinálnál:

```sql
CREATE TABLE silver.sales
USING DELTA
LOCATION '/path/to/delta_demo_output/sales_delta';
```

Ettől a metastore-ban lenne egy bejegyzés:

```text
table name: silver.sales
format: delta
location: /path/to/delta_demo_output/sales_delta
schema: order_id, customer_id, amount, order_date, source_system
```

A metastore tehát nem maga az adat. Inkább katalógus vagy telefonkönyv.

## 14. Path-alapú Delta table vs metastore table

| Megközelítés | Mit használsz? | Példa | Mire jó? |
|---|---|---|---|
| Path-alapú Delta table | Fizikai útvonal | `DeltaTable("delta_demo_output/sales_delta")` | Lokális demo, egyszerű job, közvetlen fájlkezelés |
| Metastore-ba regisztrált table | Logikai név | `silver.sales` | SQL, jogosultság, discovery, catalog, BI |

A kettő ugyanarra a fizikai Delta table-re mutathat.

## 15. Ugyanez Azure-ban hogyan nézne ki?

Lokális path helyett ADLS path lenne:

```text
abfss://silver@stcompanydata.dfs.core.windows.net/sales_delta
```

A fizikai szerkezet ugyanaz:

```text
sales_delta/
  part-0000....parquet
  _delta_log/
    00000000000000000000.json
```

Databricksben például:

```python
df.write.format("delta").mode("overwrite").save(
    "abfss://silver@stcompanydata.dfs.core.windows.net/sales_delta"
)
```

Majd catalog/metastore regisztráció:

```sql
CREATE TABLE silver.sales
USING DELTA
LOCATION 'abfss://silver@stcompanydata.dfs.core.windows.net/sales_delta';
```

## 16. ETL és ELT ugyanazzal a lokális Delta gondolkodással

Az ETL és az ELT nem fájlformátum, nem Databricks feature, és nem Azure-specifikus fogalom. Inkább arról szól, hogy **mikor történik a transzformáció**.

```text
ETL = Extract -> Transform -> Load
ELT = Extract -> Load -> Transform
```

A különbség:

| Minta | Mikor tisztítod/transzformálod az adatot? | Hova kerül először az adat? | Tipikus lakehouse réteg |
|---|---|---|---|
| **ETL** | Betöltés előtt | Már tisztított/kurált tábla | silver/gold jellegű output |
| **ELT** | Betöltés után | Először raw/bronze tábla | bronze -> silver -> gold |

A következő két mini példa ugyanazt a nyers rendelési adatot használja. Az egyik ETL-ként, a másik ELT-ként dolgozza fel.

## 17. ETL demo: előbb transzformálunk, utána töltünk Delta táblába

Ebben a mintában a forrásadat még nyers: stringként jön az összeg, a dátum és a kedvezmény is. ETL esetén először Pythonban megtisztítjuk és üzleti oszlopokat képzünk belőle, és csak a kész, strukturált adatot írjuk ki Delta table-ként.

In [ ]:
etl_source_df = pd.DataFrame({
    "order_id": ["2001", "2002", "2003", "2004"],
    "customer_id": ["C-010", "C-011", "C-010", "C-012"],
    "gross_amount_huf": ["12 900", "4 790", "25 000", "8 350"],
    "discount_pct": ["0", "10", "20", "0"],
    "order_date": ["2026-05-01", "2026-05-01", "2026-05-02", "2026-05-03"],
    "channel": ["webshop", "webshop", "partner", "pos"]
})

etl_source_df

In [ ]:
etl_clean_df = etl_source_df.copy()

etl_clean_df["order_id"] = etl_clean_df["order_id"].astype("int64")
etl_clean_df["gross_amount_huf"] = (
    etl_clean_df["gross_amount_huf"]
    .str.replace(" ", "", regex=False)
    .astype("int64")
)
etl_clean_df["discount_pct"] = etl_clean_df["discount_pct"].astype("float64")
etl_clean_df["order_date"] = pd.to_datetime(etl_clean_df["order_date"])
etl_clean_df["net_amount_huf"] = (
    etl_clean_df["gross_amount_huf"] * (1 - etl_clean_df["discount_pct"] / 100)
).round().astype("int64")
etl_clean_df["is_online"] = etl_clean_df["channel"].eq("webshop")

etl_clean_df

In [ ]:
etl_table_path = base_dir / "sales_etl_curated_delta"

write_deltalake(str(etl_table_path), etl_clean_df, mode="overwrite")

print(f"ETL output Delta table: {etl_table_path.resolve()}")
DeltaTable(str(etl_table_path)).to_pandas().sort_values("order_id")

Az ETL minta lényege itt:

```text
nyers DataFrame
  -> tisztítás és üzleti logika Pythonban
  -> kész/kurált Delta table
```

A storage-ba már olyan adat kerül, amit elemzésre vagy további feldolgozásra szántunk.

## 18. ELT demo: előbb betöltjük raw Delta táblába, utána transzformálunk

ELT esetén a nyers adatot gyorsan és kevés módosítással eltesszük egy raw/bronze táblába. Ezután ugyanabból a raw táblából készül egy tisztított silver tábla.

Ez lakehouse környezetben nagyon gyakori, mert így később újra lehet futtatni a transzformációt más szabályokkal anélkül, hogy újra ki kellene nyerni az eredeti forrásból.

In [ ]:
elt_raw_table_path = base_dir / "sales_elt_bronze_raw_delta"
elt_silver_table_path = base_dir / "sales_elt_silver_delta"

# Load lépés: a nyers adatot szinte változatlanul betöltjük bronze Delta táblába.
write_deltalake(str(elt_raw_table_path), etl_source_df, mode="overwrite")

print(f"ELT bronze/raw Delta table: {elt_raw_table_path.resolve()}")
DeltaTable(str(elt_raw_table_path)).to_pandas().sort_values("order_id")

In [ ]:
# Transform lépés: a már betöltött bronze Delta táblából olvasunk,
# és abból készítünk tisztított silver Delta táblát.
bronze_df = DeltaTable(str(elt_raw_table_path)).to_pandas()

silver_df = bronze_df.copy()
silver_df["order_id"] = silver_df["order_id"].astype("int64")
silver_df["gross_amount_huf"] = (
    silver_df["gross_amount_huf"]
    .str.replace(" ", "", regex=False)
    .astype("int64")
)
silver_df["discount_pct"] = silver_df["discount_pct"].astype("float64")
silver_df["order_date"] = pd.to_datetime(silver_df["order_date"])
silver_df["net_amount_huf"] = (
    silver_df["gross_amount_huf"] * (1 - silver_df["discount_pct"] / 100)
).round().astype("int64")
silver_df["is_online"] = silver_df["channel"].eq("webshop")

write_deltalake(str(elt_silver_table_path), silver_df, mode="overwrite")

print(f"ELT silver Delta table: {elt_silver_table_path.resolve()}")
DeltaTable(str(elt_silver_table_path)).to_pandas().sort_values("order_id")

Az ELT minta lényege itt:

```text
nyers DataFrame
  -> bronze/raw Delta table
  -> transzformáció a már betöltött adaton
  -> silver/curated Delta table
```

A bronze tábla azért értékes, mert megőrzi az eredetihez közeli állapotot. Ha később változik a `net_amount_huf` számítási logika, a silver tábla újragenerálható a bronze rétegből.

## 19. ETL vs ELT összehasonlítás a létrejött mappák alapján

Mindkét megközelítés Delta table-t hoz létre. A különbség nem a Delta formátumban van, hanem a feldolgozási sorrendben és abban, hogy megőrizzük-e külön a raw réteget.

In [ ]:
demo_tables = {
    "ETL curated": etl_table_path,
    "ELT bronze/raw": elt_raw_table_path,
    "ELT silver/curated": elt_silver_table_path,
}

for label, path in demo_tables.items():
    parquet_files = sorted(path.glob("*.parquet"))
    log_files = sorted((path / "_delta_log").glob("*.json"))
    print(f"{label}")
    print(f"  path: {path}")
    print(f"  parquet fájlok: {len(parquet_files)}")
    print(f"  delta log verziók: {len(log_files)}")
    print()

Gyakorlati döntési szabály:

- **ETL** akkor kényelmes, ha a forrásból csak a megtisztított üzleti adatot akarod megtartani.
- **ELT** akkor erős, ha fontos az auditálhatóság, az újrafeldolgozhatóság és a bronze/silver/gold lakehouse rétegezés.
- Modern data lakehouse rendszerekben gyakori az **ELT**, mert olcsóbb és hasznosabb megtartani a nyers adatot is.
- Klasszikus adattárházaknál gyakori volt az **ETL**, mert a warehouse drága volt, ezért csak előkészített adatot töltöttek be.

## 20. Landing, bronze, silver, gold rétegek

A lakehouse rétegezés arra való, hogy az adat útja ne egyetlen nagy, homályos feldolgozás legyen, hanem jól elkülönített állomások sorozata.

```text
Landing -> Bronze -> Silver -> Gold
```

| Réteg | Mit tartalmaz? | Mennyire nyers? | Tipikus cél |
|---|---|---|---|
| **Landing** | Beérkezett fájlok majdnem érintetlenül | Teljesen nyers | Forrásból érkező adat megőrzése |
| **Bronze** | Raw adat Delta táblában, ingestion metaadatokkal | Nagyon nyers | Auditálható, újraolvasható alapréteg |
| **Silver** | Tisztított, típusozott, deduplikált adat | Üzletileg használható | Elemzés és további modellezés alapja |
| **Gold** | Aggregált, üzleti célra előkészített adat | Kurált | Dashboard, riport, KPI, serving |

Fontos: ezek nem kötelező terméknevek. Inkább egy bevált gondolkodási minta. Lokálisan ugyanúgy lehet demonstrálni mappákkal és Delta táblákkal.

## 21. Landing -> Bronze -> Silver -> Gold demo

Ebben a példában először létrehozunk két nyers CSV fájlt a landing rétegben. Ez úgy viselkedik, mintha két külön export vagy két napi batch érkezett volna egy forrásrendszerből.

In [ ]:
landing_dir = base_dir / "landing" / "orders"
bronze_orders_path = base_dir / "bronze" / "orders_delta"
silver_orders_path = base_dir / "silver" / "orders_delta"
gold_daily_sales_path = base_dir / "gold" / "daily_channel_sales_delta"

landing_dir.mkdir(parents=True, exist_ok=True)

landing_batch_1 = pd.DataFrame({
    "order_id": ["3001", "3002", "3003"],
    "customer_id": ["C-020", "C-021", "C-020"],
    "gross_amount_huf": ["9900", "14900", "2490"],
    "discount_pct": ["0", "15", "0"],
    "order_date": ["2026-05-05", "2026-05-05", "2026-05-06"],
    "channel": ["webshop", "partner", "pos"]
})

landing_batch_2 = pd.DataFrame({
    "order_id": ["3004", "3005"],
    "customer_id": ["C-022", "C-023"],
    "gross_amount_huf": ["19990", "5990"],
    "discount_pct": ["10", "0"],
    "order_date": ["2026-05-06", "2026-05-07"],
    "channel": ["webshop", "webshop"]
})

landing_batch_1.to_csv(landing_dir / "orders_2026_05_05.csv", index=False)
landing_batch_2.to_csv(landing_dir / "orders_2026_05_06.csv", index=False)

for path in sorted(landing_dir.glob("*.csv")):
    print(path)

A landing rétegben még nincs Delta log, nincs táblaséma-kezelés és nincs transzformáció. Ez csak a beérkezett adat lenyomata.

In [ ]:
landing_files = sorted(landing_dir.glob("*.csv"))

bronze_frames = []
for source_file in landing_files:
    frame = pd.read_csv(source_file, dtype=str)
    frame["_source_file"] = source_file.name
    frame["_ingestion_mode"] = "batch"
    frame["_ingested_at"] = pd.Timestamp.now("UTC").isoformat()
    bronze_frames.append(frame)

bronze_orders_df = pd.concat(bronze_frames, ignore_index=True)
write_deltalake(str(bronze_orders_path), bronze_orders_df, mode="overwrite")

DeltaTable(str(bronze_orders_path)).to_pandas().sort_values("order_id")

Ez a bronze réteg: az adat még stringes és forrásközeli, de már Delta table. Emiatt van `_delta_log`, verziózható, appendelhető és visszaolvasható.

In [ ]:
bronze_orders_read_df = DeltaTable(str(bronze_orders_path)).to_pandas()

silver_orders_df = bronze_orders_read_df.copy()
silver_orders_df["order_id"] = silver_orders_df["order_id"].astype("int64")
silver_orders_df["gross_amount_huf"] = silver_orders_df["gross_amount_huf"].astype("int64")
silver_orders_df["discount_pct"] = silver_orders_df["discount_pct"].astype("float64")
silver_orders_df["order_date"] = pd.to_datetime(silver_orders_df["order_date"])
silver_orders_df["net_amount_huf"] = (
    silver_orders_df["gross_amount_huf"] * (1 - silver_orders_df["discount_pct"] / 100)
).round().astype("int64")
silver_orders_df["is_online"] = silver_orders_df["channel"].eq("webshop")

silver_orders_df = silver_orders_df.drop_duplicates(subset=["order_id"])

write_deltalake(str(silver_orders_path), silver_orders_df, mode="overwrite")

DeltaTable(str(silver_orders_path)).to_pandas().sort_values("order_id")

Ez a silver réteg: típusozott, tisztított, üzletileg már sokkal jobban használható rendelési adat.

In [ ]:
gold_daily_sales_df = (
    silver_orders_df
    .groupby(["order_date", "channel"], as_index=False)
    .agg(
        order_count=("order_id", "count"),
        gross_amount_huf=("gross_amount_huf", "sum"),
        net_amount_huf=("net_amount_huf", "sum"),
    )
    .sort_values(["order_date", "channel"])
)

write_deltalake(str(gold_daily_sales_path), gold_daily_sales_df, mode="overwrite")

DeltaTable(str(gold_daily_sales_path)).to_pandas()

Ez a gold réteg: már nem soronkénti nyers rendelés, hanem riportolásra alkalmas üzleti aggregátum.

Mentális modell:

```text
landing/orders/*.csv
  -> bronze/orders_delta        # raw, auditálható Delta
  -> silver/orders_delta        # tisztított rendelési tényadat
  -> gold/daily_channel_sales   # napi csatorna szerinti KPI
```

## 22. Fact és dimension táblák

A **fact** és **dimension** fogalmak a klasszikus adattárház-modellezésből jönnek, de lakehouse környezetben ugyanúgy használhatók, főleg a gold rétegben.

```text
Fact = mérhető események
Dimension = leíró entitások
```

| Típus | Mit tartalmaz? | Példa oszlopok | Mire jó? |
|---|---|---|---|
| **Fact table** | Üzleti események és mérőszámok | `order_id`, `customer_id`, `date_id`, `net_amount_huf` | Aggregálás, KPI, riport |
| **Dimension table** | Leíró adatok, amelyek alapján szűrünk/csoportosítunk | `customer_id`, `channel`, `date`, `month` | Kontextus, szeletelés, drill-down |

Egy rendelés tipikusan fact sor: történt egy mérhető esemény, például eladás. A vásárló, csatorna és dátum dimension jellegű: leírják, hogy kihez, hol és mikor történt az esemény.

## 23. Fact/dimension demo a silver rendelési adatból

A silver rendelési táblából készítünk egy egyszerű csillagsémát. Ezek is Delta táblák lesznek külön mappákban, de modellezési szempontból fact és dimension szerepet kapnak.

In [ ]:
dim_customer_path = base_dir / "gold" / "dim_customer_delta"
dim_channel_path = base_dir / "gold" / "dim_channel_delta"
dim_date_path = base_dir / "gold" / "dim_date_delta"
fact_sales_path = base_dir / "gold" / "fact_sales_delta"

silver_for_model_df = DeltaTable(str(silver_orders_path)).to_pandas()
silver_for_model_df["order_date"] = pd.to_datetime(silver_for_model_df["order_date"])

customer_names = {
    "C-020": "Kovacs Anna",
    "C-021": "Nagy Bela",
    "C-022": "Toth Csilla",
    "C-023": "Szabo David",
}

dim_customer_df = (
    silver_for_model_df[["customer_id"]]
    .drop_duplicates()
    .sort_values("customer_id")
    .assign(customer_name=lambda df: df["customer_id"].map(customer_names).fillna("Unknown customer"))
)

dim_customer_df

In [ ]:
channel_labels = {
    "webshop": "Online webshop",
    "partner": "Partner sales",
    "pos": "Physical store",
}

dim_channel_df = (
    silver_for_model_df[["channel"]]
    .drop_duplicates()
    .sort_values("channel")
    .assign(channel_name=lambda df: df["channel"].map(channel_labels))
)

dim_channel_df

In [ ]:
dim_date_df = (
    silver_for_model_df[["order_date"]]
    .drop_duplicates()
    .sort_values("order_date")
    .assign(
        date_id=lambda df: df["order_date"].dt.strftime("%Y%m%d").astype("int64"),
        year=lambda df: df["order_date"].dt.year,
        month=lambda df: df["order_date"].dt.month,
        day=lambda df: df["order_date"].dt.day,
    )
    [["date_id", "order_date", "year", "month", "day"]]
)

dim_date_df

In [ ]:
fact_sales_df = (
    silver_for_model_df
    .merge(dim_date_df[["date_id", "order_date"]], on="order_date", how="left")
    [[
        "order_id",
        "customer_id",
        "channel",
        "date_id",
        "gross_amount_huf",
        "discount_pct",
        "net_amount_huf",
        "is_online",
    ]]
    .sort_values("order_id")
)

fact_sales_df

In [ ]:
write_deltalake(str(dim_customer_path), dim_customer_df, mode="overwrite")
write_deltalake(str(dim_channel_path), dim_channel_df, mode="overwrite")
write_deltalake(str(dim_date_path), dim_date_df, mode="overwrite")
write_deltalake(str(fact_sales_path), fact_sales_df, mode="overwrite")

for label, path in {
    "dim_customer": dim_customer_path,
    "dim_channel": dim_channel_path,
    "dim_date": dim_date_path,
    "fact_sales": fact_sales_path,
}.items():
    print(f"{label}: {path}")

A létrejött modell:

```text
                  dim_customer
                       ?
dim_date  <-  fact_sales  ->  dim_channel
```

A `fact_sales` tartalmazza a mérhető üzleti eseményeket: rendelések, összegek, kedvezmény, nettó érték. A dimension táblák adják hozzá a leíró kontextust: vásárló, csatorna, dátum.

In [ ]:
sales_mart_df = (
    fact_sales_df
    .merge(dim_customer_df, on="customer_id", how="left")
    .merge(dim_channel_df, on="channel", how="left")
    .merge(dim_date_df, on="date_id", how="left")
)

sales_mart_df[[
    "order_id",
    "customer_name",
    "channel_name",
    "order_date",
    "net_amount_huf",
]]

In [ ]:
customer_kpi_df = (
    sales_mart_df
    .groupby("customer_name", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_net_amount_huf=("net_amount_huf", "sum"),
    )
    .sort_values("total_net_amount_huf", ascending=False)
)

customer_kpi_df

Miért hasznos ez?

- A fact tábla általában hosszú és eseményorientált.
- A dimension táblák általában kisebbek, leíró jellegűek.
- Riportoláskor a fact táblát össze lehet kapcsolni a dimension táblákkal.
- A gold rétegben gyakori, hogy már ilyen riportbarát fact/dimension modell jelenik meg.

Egyszerű mondatban: **a fact az, amit mérsz; a dimension az, ami alapján értelmezed, szűröd vagy csoportosítod.**

## 24. Batch feldolgozás

Batch feldolgozásnál egyszerre egy véges adatcsomagot dolgozol fel. Például: minden éjjel 01:00-kor beolvasod az előző napi CSV-ket, frissíted a bronze, silver és gold rétegeket, majd kész.

A fenti landing -> bronze -> silver -> gold példa batch jellegű volt, mert az összes landing fájlt egyszerre olvastuk be.

In [ ]:
batch_summary = pd.DataFrame({
    "step": ["extract", "load bronze", "transform silver", "aggregate gold"],
    "input": ["landing CSV files", "landing DataFrame", "bronze Delta", "silver Delta"],
    "output": ["pandas DataFrame", "bronze Delta", "silver Delta", "gold Delta"],
    "execution_style": ["batch", "batch", "batch", "batch"]
})

batch_summary

Batch esetén a pipeline tipikus kérdései:

- Melyik fájlokat dolgoztuk már fel?
- Mi történik, ha újra kell futtatni ugyanazt a napot?
- Felülírjuk a célréteget, appendelünk, vagy merge/upsert kell?
- Hogyan kezeljük a késve érkező adatot?

## 25. Streaming és micro-batch gondolkodás

Streamingnél az adat nem egy nagy, lezárt csomagként érkezik, hanem folyamatosan vagy kis adagokban. Valódi streaminghez általában Spark Structured Streaming, Flink, Kafka, Event Hubs vagy hasonló eszköz kell.

Ebben a lokális notebookban nem indítunk valódi streaming rendszert. Helyette micro-batch szimulációt csinálunk: több kis adatcsomagot dolgozunk fel egymás után, mintha folyamatosan érkeznének az események.

In [ ]:
stream_bronze_path = base_dir / "bronze" / "orders_stream_delta"
stream_silver_path = base_dir / "silver" / "orders_stream_delta"
stream_gold_path = base_dir / "gold" / "orders_stream_kpi_delta"

stream_batches = [
    pd.DataFrame({
        "order_id": ["4001", "4002"],
        "customer_id": ["C-030", "C-031"],
        "gross_amount_huf": ["12900", "3490"],
        "discount_pct": ["0", "0"],
        "order_date": ["2026-05-08", "2026-05-08"],
        "channel": ["webshop", "pos"]
    }),
    pd.DataFrame({
        "order_id": ["4003", "4004"],
        "customer_id": ["C-032", "C-030"],
        "gross_amount_huf": ["8490", "22990"],
        "discount_pct": ["5", "20"],
        "order_date": ["2026-05-08", "2026-05-09"],
        "channel": ["partner", "webshop"]
    })
]

stream_batches

In [ ]:
def transform_orders(raw_df):
    result = raw_df.copy()
    result["order_id"] = result["order_id"].astype("int64")
    result["gross_amount_huf"] = result["gross_amount_huf"].astype("int64")
    result["discount_pct"] = result["discount_pct"].astype("float64")
    result["order_date"] = pd.to_datetime(result["order_date"])
    result["net_amount_huf"] = (
        result["gross_amount_huf"] * (1 - result["discount_pct"] / 100)
    ).round().astype("int64")
    result["is_online"] = result["channel"].eq("webshop")
    return result.drop_duplicates(subset=["order_id"])


def aggregate_daily_channel_sales(silver_df):
    return (
        silver_df
        .groupby(["order_date", "channel"], as_index=False)
        .agg(
            order_count=("order_id", "count"),
            gross_amount_huf=("gross_amount_huf", "sum"),
            net_amount_huf=("net_amount_huf", "sum"),
        )
        .sort_values(["order_date", "channel"])
    )


def process_micro_batch(batch_df, batch_id):
    bronze_batch_df = batch_df.copy()
    bronze_batch_df["_batch_id"] = batch_id
    bronze_batch_df["_ingestion_mode"] = "micro-batch"
    bronze_batch_df["_ingested_at"] = pd.Timestamp.now("UTC").isoformat()

    bronze_mode = "overwrite" if batch_id == 1 else "append"
    write_deltalake(str(stream_bronze_path), bronze_batch_df, mode=bronze_mode)

    all_bronze_df = DeltaTable(str(stream_bronze_path)).to_pandas()
    all_silver_df = transform_orders(all_bronze_df)
    all_gold_df = aggregate_daily_channel_sales(all_silver_df)

    write_deltalake(str(stream_silver_path), all_silver_df, mode="overwrite")
    write_deltalake(str(stream_gold_path), all_gold_df, mode="overwrite")

    print(f"micro-batch {batch_id} feldolgozva")
    print(f"  bronze sorok: {len(all_bronze_df)}")
    print(f"  silver sorok: {len(all_silver_df)}")
    print(f"  gold sorok: {len(all_gold_df)}")


for batch_id, batch_df in enumerate(stream_batches, start=1):
    process_micro_batch(batch_df, batch_id)

In [ ]:
from IPython.display import display

print("Bronze stream tábla:")
display(DeltaTable(str(stream_bronze_path)).to_pandas().sort_values("order_id"))

print("Gold stream KPI tábla:")
DeltaTable(str(stream_gold_path)).to_pandas()

A micro-batch szimuláció lényege:

```text
kis eseménycsomag 1 -> append bronze -> újraszámol silver/gold
kis eseménycsomag 2 -> append bronze -> újraszámol silver/gold
...
```

Valódi streamingnél ugyanez a gondolat futna folyamatosan, checkpointtal, triggerrel és állapotkezeléssel. A lakehouse rétegek viszont ugyanazok maradnak: raw események bronze-ban, tisztított események silverben, fogyasztásra kész KPI-k goldban.

## 26. Batch vs streaming röviden

| Szempont | Batch | Streaming / micro-batch |
|---|---|---|
| Adat érkezése | Véges csomagokban | Folyamatosan vagy kis adagokban |
| Tipikus gyakoriság | Óránként, naponta, hetente | Másodpercenként, percenként, folyamatosan |
| Egyszerűség | Egyszerűbb fejleszteni és újrafuttatni | Több állapotkezelés, checkpoint, késő adat |
| Jó választás, ha | Nem kell azonnali frissesség | Közel valós idejű adat kell |
| Példa | Napi sales riport | Élő rendelésmonitoring, fraud detection |

Egy fontos gyakorlati mondat: sok modern streaming rendszer valójában **micro-batch** módon működik. Ez azt jelenti, hogy nem egyetlen végtelen Python ciklust írunk kézzel, hanem a feldolgozó engine kis, ismétlődő batch-ekre bontja a folyamatos adatfolyamot.

## 27. Open source metastore/catalog toolok kipróbáláshoz

Ha ezt a notebookot tovább akarod vinni a `path alapján olvasom` világból a `catalog.schema.table néven hivatkozom rá` világba, akkor ezekkel az open source toolokkal érdemes kísérletezni.

| Tool | Mit ad hozzá? | Formátum fókusz | Mire jó tanuláskor? |
|---|---|---|---|
| **Hive Metastore** | Klasszikus Spark SQL metastore: database/table nevek, schema, location | Delta, Parquet, Hive, részben Iceberg/Hudi integrációval | Megérteni az alap metastore modellt: a tábla neve csak metadata, az adat továbbra is fájlokban van |
| **Unity Catalog OSS** | Databricks-szerű catalog modell: `catalog.schema.table`, jogosultságok és egységes namespace | Delta Lake, Iceberg, Parquet és más asset típusok | Kipróbálni, hogyan néz ki a Unity Catalog gondolkodás Databricks nélkül |
| **Apache Gravitino** | Federált metadata réteg, egyfajta catalog of catalogs | Több engine és több catalog típus | Megérteni, hogyan lehet több adattárolót és catalogot egy közös metadata réteg alá szervezni |
| **Project Nessie** | Git-szerű catalog branch/tag/commit szemlélettel | Főleg Apache Iceberg | Kipróbálni a data lake táblák verziózását catalog szinten, nem csak egy tábla Delta logján belül |
| **Apache Polaris** | Iceberg REST catalog | Apache Iceberg | Megérteni a modern Iceberg REST Catalog mintát, több query engine közös catalogjaként |

A mostani Delta példához a legközelebbi két út:

```text
1. Spark + Hive Metastore
   sales_delta mappa regisztrálása táblanévként

2. Unity Catalog OSS
   Databricks-szerű catalog/schema/table modell kipróbálása lokálisan
```

A Hive Metastore esetén a mentális modell nagyon egyszerű:

```text
demo.sales név
  -> format: delta
  -> location: delta_demo_output/sales_delta
  -> schema: order_id, customer_id, amount, order_date, source_system
```

Vagyis a catalog/metastore nem másolja be az adatot. Csak nyilvántartja, hogy egy logikai táblanév melyik fizikai Delta table locationre mutat.

Gyakorlati választás:

- Ha a **metastore alapfogalmat** akarod megérteni: **Hive Metastore**.
- Ha a **Databricks Unity Catalog modelljét** akarod megérteni: **Unity Catalog OSS**.
- Ha **Iceberges modern catalogot** akarsz kipróbálni: **Polaris** vagy **Nessie**.
- Ha **több catalog és több rendszer fölé** akarsz közös réteget: **Apache Gravitino**.

## 28. Összefoglalás

A legfontosabb mondatok:

- Delta table-t létre tudsz hozni Databricks és Synapse nélkül is.
- Ehhez kell egy Delta-kompatibilis writer, például `deltalake` vagy `delta-spark`.
- A Delta table fizikailag adatfájlokból és `_delta_log` naplóból áll.
- A schema a Delta metadata része.
- A metastore csak akkor kell, ha névvel, SQL-ből, catalogként akarod használni.
- Azure-ban ugyanez ADLS/Storage Account path-on történik, csak nem lokális mappában.
- A landing/bronze/silver/gold rétegek az adat feldolgozottsági állapotát írják le.
- A batch és streaming közti fő különbség az, hogy véges csomagokat vagy folyamatosan érkező adatot dolgozol fel.
- A fact táblák mérhető eseményeket, a dimension táblák leíró entitásokat tartalmaznak.